# Maps

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Store and look up values by key with a Map, counting and grouping along the way.
- Go through a map's entries, and choose HashMap, LinkedHashMap, TreeMap or EnumMap by the key order you need.

**In this notebook**, one idea at a time (run each cell before moving on):
1. Map: looking things up by key
2. Counting with a map
3. Key order: Hash, Linked or Tree
4. A map of lists
5. EnumMap, and your own types as keys
6. Which one to use?

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## 1. Map: looking things up by key

A **Map** links each **key** to a **value**, like a dictionary links a word to its meaning. `put(key, value)` stores (and replaces the old value if the key was there), `get(key)` reads (`null` when the key is missing), `getOrDefault(key, x)` gives `x` instead of `null`, and `containsKey` asks. To go through it, loop over `entrySet()`: each entry has `getKey()` and `getValue()`.

In [ ]:
Map<String, Double> prices = new HashMap<>();
prices.put("apple", 0.5);
prices.put("bread", 2.25);
prices.put("apple", 0.6);          // replaces 0.5
IO.println(prices.get("apple"));
IO.println(prices.get("milk"));    // not there
IO.println(prices.getOrDefault("milk", 0.0));

List<String> cart = List.of("apple", "bread", "apple", "milk");
double total = 0;
for (String item : cart) {
    total += prices.getOrDefault(item, 0.0);
}
IO.println("Total: " + total);
for (Map.Entry<String, Double> e : prices.entrySet()) {
    IO.println(e.getKey() + " costs " + e.getValue());
}

🧪 **Try it:** add milk to the price list and run again. Then use `TreeMap` instead of `HashMap`: what happens to the order of the last loop?

### ✏️ Exercise: Word lengths

Build a map from each word to its length. A repeated word just gets the same entry again.

In [ ]:
Map<String, Integer> lengths(List<String> words) {
    // TODO: put each word with its length
    return new HashMap<>();
}

void main() {
    check("two words", lengths(List.of("hi", "there")), Map.of("hi", 2, "there", 5));
    check("repeated", lengths(List.of("a", "a")), Map.of("a", 1));
    check("empty", lengths(List.of()), Map.of());
}

<details>
<summary>Show an answer</summary>

```java
Map<String, Integer> lengths(List<String> words) {
    Map<String, Integer> result = new HashMap<>();
    for (String w : words) {
        result.put(w, w.length());
    }
    return result;
}

void main() {
    check("two words", lengths(List.of("hi", "there")), Map.of("hi", 2, "there", 5));
    check("repeated", lengths(List.of("a", "a")), Map.of("a", 1));
    check("empty", lengths(List.of()), Map.of());
}
```

</details>

## 2. Counting with a map

A very common job: count how many times each thing appears. For each item, add 1 to its count, starting at 0 for a new key. `merge(key, 1, Integer::sum)` does it in one line: it puts 1 for a new key, or adds 1 to the old value. The slides follow it.

#### 🎞️ Counting with a map

![merge adds each new key with 1 and adds 1 to a key that is already there; get returns null for a missing key, while getOrDefault returns the default](map-steps.svg)

In [ ]:
List<String> votes = List.of("red", "blue", "red", "green", "red", "blue");
Map<String, Integer> count = new HashMap<>();
for (String v : votes) {
    count.put(v, count.getOrDefault(v, 0) + 1);   // the long way
}
IO.println(count);
Map<String, Integer> count2 = new HashMap<>();
for (String v : votes) {
    count2.merge(v, 1, Integer::sum);               // the short way
}
IO.println(count2.equals(count));

🧪 **Try it:** count the letters of a word instead of the votes: go through `word.toCharArray()` with a `Map<Character, Integer>`.

## 3. Key order: Hash, Linked or Tree

Like sets, maps come in three orders. `HashMap` is fastest but its order looks random; `LinkedHashMap` keeps the order keys were added; `TreeMap` keeps keys sorted and can answer "the first key", "the keys before X".

In [ ]:
String[] countries = {"Peru", "Brazil", "Chile", "Angola"};
Map<String, Integer> hash = new HashMap<>();
Map<String, Integer> linked = new LinkedHashMap<>();
TreeMap<String, Integer> tree = new TreeMap<>();
for (String c : countries) {
    hash.put(c, c.length());
    linked.put(c, c.length());
    tree.put(c, c.length());
}
IO.println("Hash:   " + hash);
IO.println("Linked: " + linked);
IO.println("Tree:   " + tree);
IO.println("first: " + tree.firstKey() + ", before C: " + tree.headMap("C"));

🧪 **Try it:** add "Benin" to the countries and see where it lands in each map. Then print `tree.lastKey()`.

## 4. A map of lists

A value can be a collection: `Map<String, List<String>>` gives each key a whole list. `computeIfAbsent(key, k -> new ArrayList<>())` returns the key's list, creating an empty one the first time, so you can add to it right away.

In [ ]:
String[][] classes = {{"Ana", "Java"}, {"Bia", "C"}, {"Caio", "Java"}, {"Davi", "Java"}};
Map<String, List<String>> students = new TreeMap<>();
for (String[] row : classes) {
    students.computeIfAbsent(row[1], k -> new ArrayList<>()).add(row[0]);
}
IO.println(students);
IO.println(students.get("Java").size() + " in Java");

🧪 **Try it:** replace `computeIfAbsent(...)` with `students.get(row[1]).add(row[0])` and run: the first `get` of each class is `null`.

### ✏️ Exercise: Words by first letter

Build a map from each first letter to the words that start with it, in their original order.

In [ ]:
Map<Character, List<String>> byFirstLetter(List<String> words) {
    // TODO: computeIfAbsent(w.charAt(0), …).add(w)
    return new HashMap<>();
}

void main() {
    check("three words", byFirstLetter(List.of("ant", "bee", "asp")), Map.of('a', List.of("ant", "asp"), 'b', List.of("bee")));
    check("no words", byFirstLetter(List.of()), Map.of());
}

<details>
<summary>Show an answer</summary>

```java
Map<Character, List<String>> byFirstLetter(List<String> words) {
    Map<Character, List<String>> result = new HashMap<>();
    for (String w : words) {
        result.computeIfAbsent(w.charAt(0), k -> new ArrayList<>()).add(w);
    }
    return result;
}

void main() {
    check("three words", byFirstLetter(List.of("ant", "bee", "asp")), Map.of('a', List.of("ant", "asp"), 'b', List.of("bee")));
    check("no words", byFirstLetter(List.of()), Map.of());
}
```

</details>

## 5. EnumMap, and your own types as keys

When the keys are the values of an enum, an **EnumMap** is the natural choice: fast, and always in the enum's order. Your own classes can be keys too, as long as `equals` and `hashCode` compare values: records already do.

In [ ]:
enum Size { SMALL, MEDIUM, LARGE }

record Seat(char row, int number) {}

void main() {
    Map<Size, Integer> sold = new EnumMap<>(Size.class);
    sold.merge(Size.LARGE, 2, Integer::sum);
    sold.merge(Size.SMALL, 5, Integer::sum);
    sold.merge(Size.LARGE, 1, Integer::sum);
    IO.println(sold);                         // in the enum's order

    Map<Seat, String> booked = new HashMap<>();
    booked.put(new Seat('A', 1), "Ana");
    IO.println(booked.get(new Seat('A', 1)));   // an equal record finds it
}

🧪 **Try it:** turn `record Seat` into a plain class with a constructor and run: the lookup gives `null`, because the class compares references.

## 6. Which one to use?

| You need… | Use |
|---|---|
| items in order, repeats allowed, access by position | `List` (`ArrayList`) |
| each item once, fast "is it there?" | `Set` (`HashSet`; `LinkedHashSet` for arrival order, `TreeSet` for sorted) |
| a value for each key | `Map` (`HashMap`; `LinkedHashMap`, `TreeMap`) |
| first in, first out, or last in, first out | `ArrayDeque` as a queue or a stack |

**Watch out:** `map.get(key)` gives `null` for a missing key, and using that `null` as a number stops the program with a `NullPointerException`. Prefer `getOrDefault`.

In [ ]:
Map<String, Integer> stock = new HashMap<>();
stock.put("pens", 12);
Integer pencils = stock.get("pencils");
IO.println(pencils);                              // null: no such key
IO.println(stock.getOrDefault("pencils", 0) + 5); // safe

🧪 **Try it:** write `int n = stock.get("pencils");` and run: that's the NullPointerException. Then change it to use `getOrDefault`.

## In short

A **Map** links each **key** to a value, like a dictionary links a word to its meaning. Each key appears once: putting it again replaces its value.

```java
void main() {
    Map<String, Integer> votes = new HashMap<>();
    for (String v : List.of("red", "blue", "red", "green", "red")) {
        votes.merge(v, 1, Integer::sum);       // add 1, starting at 1
    }
    IO.println(votes.get("red"));               // 3
    IO.println(votes.getOrDefault("pink", 0));  // 0: no such key

Map<String, Integer> sorted = new TreeMap<>(votes);
    for (Map.Entry<String, Integer> e : sorted.entrySet()) {
        IO.println(e.getKey() + ": " + e.getValue());
    }
}
```

`HashMap` has no useful key order, `LinkedHashMap` keeps insertion order, `TreeMap` keeps keys sorted, and `EnumMap` follows an enum's order.

**Watch out:** `get` returns `null` for a missing key, and using that `null` as a number crashes. Prefer `getOrDefault`.

## Readings

- [Using Maps to Store Key Value Pairs](https://dev.java/learn/api/collections-and-streams/collections-framework/maps/) (dev.java (Oracle), 9 min): How a map stores key/value pairs, then "Practicing Map Operations": getting a value, getOrDefault, checking for a key, and removing an entry.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Word Frequency (HashMap)** · Medium
- **Phone Book (TreeMap)** · Medium
- **Weekly Study Hours (EnumMap)** · Medium
- **Most Common Words** · Hard